# Act 1 — Who's actually asking?

Every action on GCP is some principal trying to do something to some resource. Before you can talk about what they're allowed to do, you have to pin down who they are and where they live. That's the job of this act.

GCP has more principal types than most clouds. Some are humans; some are workloads; some are bags of humans (groups); some are *everyone on the internet*. Each type sits in a different place and gets fed into IAM the same way.

## Cloud Identity and Workspace — the directory

GCP needs a directory of humans to grant access to. Two products do this, and they are nearly the same thing:

- **Google Workspace** — the paid productivity suite (Gmail, Drive, Calendar) bundled with a user directory.
- **Cloud Identity** — the directory **without** the productivity apps. Free at the basic tier, paid for premium features.

Either way, you end up with a domain (`acme.com`) and a set of users (`alice@acme.com`). Those users are what GCP IAM bindings reference. The Organization node we met in notebook 01 is bound to one of these directories at create time.

**Hybrid sources** plug into the directory:

- **Google Cloud Directory Sync (GCDS)** pulls users and groups from on-prem Active Directory or LDAP.
- **Okta / Ping / Azure AD via SAML** federates external IdPs for sign-in (the user still appears as `alice@acme.com` to IAM).
- **Workforce Identity Federation** is the newer way to let external IdP users access GCP without provisioning them into Cloud Identity at all — Workforce pool subjects get principal IDs like `principal://iam.googleapis.com/.../subject/alice`.

**Compare:** Cloud Identity ≈ Microsoft Entra ID's free tier ≈ AWS IAM Identity Center (without the AWS-specific assume-role glue). The shape is recognisable from either.

## Member types

Any IAM binding has a `members` list — the principals it grants the role to. Seven member types matter in practice:

| Member type | Example | What it is |
|---|---|---|
| `user:` | `user:alice@acme.com` | A single human in Cloud Identity / Workspace |
| `group:` | `group:platform-team@acme.com` | A group in Cloud Identity, transitively expanded |
| `serviceAccount:` | `serviceAccount:deployer@acme-prod.iam.gserviceaccount.com` | A workload identity owned by a project |
| `domain:` | `domain:acme.com` | Everyone in the directory — rarely the right answer |
| `allAuthenticatedUsers` | (no qualifier) | Any Google account in the world — dangerous |
| `allUsers` | (no qualifier) | Public, no authentication — only for genuinely public buckets/services |
| `principalSet://` | Workforce/Workload Identity Federation subjects | External IdP users or external workload tokens |

**Grant to groups, not users.** A binding to `group:platform-team@acme.com` follows the team as members come and go; a binding to `user:alice@acme.com` becomes orphaned the moment Alice changes roles. This is the single highest-leverage IAM habit.

# Act 2 — What are they allowed to do?

A principal alone does nothing. You give them a **role** at a **resource** (or some ancestor in the hierarchy), and the role's permissions apply. The whole game of GCP IAM is choosing the right role at the right node.

Three kinds of roles exist. Two are pre-made; one you write yourself. Most engineers default to the second kind and never need the third.

## Three kinds of roles

| Kind | Examples | When to use |
|---|---|---|
| **Primitive** (basic) | `roles/owner`, `roles/editor`, `roles/viewer` | Almost never — too coarse for production. Acceptable for sandbox projects. |
| **Predefined** | `roles/run.developer`, `roles/storage.objectViewer`, `roles/bigquery.dataEditor` | The default. Google maintains hundreds, scoped per service. |
| **Custom** | `roles/acme.invoiceProcessor` | When no predefined role is narrow enough. Maintained by you. |

A role is a named bundle of **permissions** — strings like `storage.objects.get`, `run.services.create`, `bigquery.tables.update`. Permissions are 1:1 with API methods. You almost never bind permissions directly; you bind roles, and the role expands to its permissions at evaluation time.

**The primitive-role warning is real.** `roles/owner` includes permission to grant `roles/owner` to anyone else (`resourcemanager.projects.setIamPolicy`). One leaked owner binding compromises the project's entire access surface. Treat owner like root.

## The policy model — bindings

An IAM **policy** attached to a resource is a list of **bindings**. A binding is a tuple:

```
binding {
  role:      "roles/run.developer"
  members:   ["group:app-team@acme.com"]
  condition: optional CEL expression
}
```

Each resource has one policy, with one or more bindings. The policy is replaced atomically when you change it — you cannot "add a binding" without reading-modifying-writing the whole policy. The CLI and SDK hide this with `add-iam-policy-binding` helpers, but it's worth knowing the underlying model.

A principal's **effective permissions** on a resource are: union of all roles granted to them by any binding on this resource or any ancestor, filtered by IAM Conditions and any Deny policies. We come back to Deny in Act 4.

## IAM Conditions — CEL on bindings

A binding can carry a **condition** — a CEL (Common Expression Language) expression that must evaluate to `true` for the binding to apply.

Typical uses:

- **Time-bound access** — `request.time < timestamp("2026-12-31T23:59:59Z")` to expire a JIT grant.
- **Resource-name prefix** — `resource.name.startsWith("projects/_/buckets/acme-public-")` to restrict a binding to a subset of buckets by name.
- **IP / device gates** — limit access to corporate IP ranges or trusted devices via Access Context Manager.

Conditions are evaluated **per request** at allow time. They're the closest GCP equivalent to AWS IAM policy `Condition` blocks, though more constrained: the available surface is the request context (`request.time`, `request.auth`), the resource (`resource.name`, `resource.type`, `resource.service`), and a handful of Access Context Manager attributes.

Not every role supports conditions — predefined roles must be marked as condition-eligible (most modern ones are). The error you get when you try to attach a condition to an ineligible role is explicit.

In [ ]:
# Adding a conditional IAM binding to a project programmatically.
# Note the read-modify-write pattern — the policy is replaced atomically.
from google.cloud import resourcemanager_v3
from google.iam.v1 import policy_pb2

PROJECT_ID = "acme-app-prod"

client = resourcemanager_v3.ProjectsClient()
resource = f"projects/{PROJECT_ID}"

policy = client.get_iam_policy(request={"resource": resource})
policy.bindings.append(policy_pb2.Binding(
    role="roles/run.developer",
    members=["group:app-team@acme.com"],
    condition=policy_pb2.Expr(
        title="prod access window",
        expression='request.time < timestamp("2026-12-31T23:59:59Z")',
    ),
))
client.set_iam_policy(request={"resource": resource, "policy": policy})

# Act 3 — Service Accounts and Workload Identity Federation

Humans aren't the only principals on GCP. The harder case is *workloads* — code running somewhere that needs to call a GCP API. A Cloud Run service writing to GCS. A GKE pod querying BigQuery. A GitHub Actions job deploying a new Cloud Run revision.

The answer to all three is the same primitive: a **service account**. What differs is *how the workload proves it's allowed to use that service account*. That second question is where Workload Identity Federation comes in, and it's GCP's most distinctive identity feature.

## Service Accounts — workload principals

A **service account** (SA) is a special principal owned by a project. It has an email like `deployer@acme-prod.iam.gserviceaccount.com` and a numeric unique ID. Three things to internalise:

1. **An SA can be a *principal*** — you bind roles to it, and it can act on resources. `serviceAccount:deployer@acme-prod.iam.gserviceaccount.com` appears in `members` lists exactly like a user does.
2. **An SA is *also* a resource** — you bind roles *on* it (e.g. `roles/iam.serviceAccountUser`) to control who can act *as* it. This is the impersonation surface.
3. **An SA can have JSON keys**, but you almost never should create one. Long-lived SA keys are the leading cause of GCP credential incidents. The modern default is `iam.disableServiceAccountKeyCreation` enforced at the Org Policy level.

**Attached vs impersonated:**

- **Attached** — when you create a Compute Engine VM, Cloud Run service, Cloud Function, or GKE workload, you *attach* an SA to it. The workload picks up that SA's credentials automatically via the metadata server. No keys, no config — the platform mints short-lived tokens on demand.
- **Impersonated** — a different principal (a user, or another SA) calls `iam.serviceAccounts.generateAccessToken` to get a short-lived token *as* the target SA. This is how CI/CD pipelines, on-call human break-glass, and cross-project workflows all work.

The impersonation pattern is the canonical way to give *anything* access to a target SA without keys. Workload Identity Federation, next, is impersonation triggered by an external identity proof.

## Workload Identity Federation — two flavours, two-step dance

WIF lets you grant GCP access to identities that live *outside* GCP — without provisioning users in Cloud Identity and without minting SA keys.

Two flavours:

- **Workforce Identity Federation** — for *humans* signing in from an external IdP (Okta, Azure AD, Ping).
- **Workload Identity Federation** — for *workloads* outside GCP (GitHub Actions, AWS IAM roles, Azure Service Principals, on-prem K8s, any OIDC- or SAML-capable identity).

Both flow through a **Workload Identity Pool** that contains one or more **Providers**, each trusting an external IdP and mapping its claims into Google-side attributes via `attribute_mapping`, with `attribute_condition` (a CEL expression) as the gate that decides which external tokens are even accepted.

**The two-step token dance** is what distinguishes GCP from AWS and Azure. AWS does `AssumeRoleWithWebIdentity` — one call, one result. Azure does an OAuth token exchange at `login.microsoftonline.com` — one call, one result. GCP splits federation and impersonation into two calls:

1. **STS token exchange** — `POST https://sts.googleapis.com/v1/token` with the external token (e.g. a GitHub OIDC ID token) plus the WIF audience. The response is a **federated access token** representing the external identity, not yet a Google identity.
2. **`generateAccessToken`** — call `iam.googleapis.com/.../:generateAccessToken` against a target SA, presenting the federated token. The response is a short-lived **SA access token** that the workload uses to call GCP APIs.

Step 2 requires the federated identity to have `roles/iam.workloadIdentityUser` *on the target SA*. That's the gate that connects "this external identity is real" to "this external identity can impersonate this specific SA."

For GitHub Actions the canonical pattern is `google-github-actions/auth@v2`, which does both steps for you behind a single `with: workload_identity_provider:` config. Underneath, it's still the two-step dance.

## Workload Identity in GKE — Kubernetes SAs ↔ Google SAs

GKE has a related but distinct feature called **Workload Identity** (no "Federation" suffix). It lets a *Kubernetes* SA in your cluster impersonate a *Google* SA, so a pod gets short-lived Google credentials by annotation rather than by mounting an SA key.

The binding is two-sided:

- **On the GCP side:** grant `roles/iam.workloadIdentityUser` on the Google SA to a principal of the form `serviceAccount:PROJECT.svc.id.goog[NAMESPACE/KSA-NAME]`.
- **On the Kubernetes side:** annotate the KSA with `iam.gke.io/gcp-service-account: GSA-EMAIL`.

A pod using that KSA picks up the Google SA's identity via the GKE metadata server. No SA JSON keys, no mounted secrets, no rotation. This is the default for new GKE clusters (Autopilot enforces it).

# Act 4 — Guardrails above IAM

IAM tells you *what a principal is allowed to do*. It doesn't tell you what *can exist* in a project in the first place, or what the network perimeter looks like, or which actions are *banned regardless of any IAM grant*. Those questions belong to three guardrails that sit above IAM in the hierarchy.

This act is where the GCP-unique answer to enterprise governance lives. AWS folds most of this into SCPs at the Organization level; GCP splits it into three independently-targeted layers.

## Organization Policy

An **Organization Policy** is a constraint applied to an Organization, folder, or project that restricts *which configurations are even allowed to exist*. Two flavours of constraint:

- **Boolean** — on/off. `iam.disableServiceAccountKeyCreation = true` blocks creation of SA JSON keys anywhere under that node.
- **List** — allow-list or deny-list of values. `compute.allowedTrustedImageProjects` constrains which projects' VM images can be used for boot disks. `gcp.resourceLocations` constrains which regions resources can be created in.

Constraints **cascade down** the hierarchy and **stack** with ancestors. Setting `gcp.resourceLocations = ["in:eu-locations"]` at a `eu-prod` folder forces every project under it to deploy only to E-U regions, regardless of what individual project owners try.

Common constraints to apply early:

- `iam.disableServiceAccountKeyCreation` — no SA keys.
- `iam.automaticIamGrantsForDefaultServiceAccounts` — disable the legacy auto-grant of `roles/editor` to default SAs.
- `compute.requireOsLogin` — force SSH via IAM-managed OS Login.
- `compute.vmExternalIpAccess` — restrict which VMs can have public IPs.
- `storage.uniformBucketLevelAccess` — force uniform IAM on new buckets (no ACL legacy).

Org Policy is *what can exist*. IAM is *who can act on what exists*. They're orthogonal.

## Hierarchical Firewall Policies

VPC firewall rules live inside a VPC and apply to that VPC's instances. But you often want rules that apply *across* every VPC in every project in a folder or organization — e.g. "never allow inbound SSH from the internet, anywhere."

That's what **Hierarchical Firewall Policies** are for. They attach to an Organization or folder node and evaluate *before* VPC firewall rules. A `deny` at the Org level applies to every VPC in every project below it.

Two rule actions worth knowing:

- **`goto_next`** — explicitly fall through to the next policy in the chain (the folder's policy, then the project's VPC rules).
- **`deny`** / **`allow`** — terminal at this policy.

Use hierarchical firewall for non-negotiable network rules: "no public SSH," "no egress to known-bad CIDRs," "allow only corporate VPN ranges on management ports."

## IAM Deny policies — the explicit deny layer

GCP IAM is fundamentally **allow-only**: a principal can do something if and only if some binding grants the permission. There's no "deny" inside a regular binding.

That default is the right behaviour 95% of the time but breaks down for a few enterprise cases:

- "Nobody, including Owners, can delete the audit log sink."
- "Service accounts in `acme-finance-prod` may never grant themselves additional bindings."
- "The break-glass role can only be used between 02:00 and 04:00 UTC."

For these, GCP added **Deny policies** — a separate policy attached to an Organization, folder, or project that lists `(deniedPrincipals, deniedPermissions, optional CEL condition)` tuples. A deny policy match **overrides any allow**, including project owner. Evaluation order: deny is checked first; if any deny matches, the request fails immediately.

**The contrast vs AWS is the key callout.** AWS IAM is *deny-wins by default*: every policy statement is `Allow` or `Deny`, and any `Deny` anywhere in the union of policies kills the request. GCP doesn't put `Deny` inside ordinary bindings — explicit deny is a separate, deliberately rarer mechanism. The practical effect: GCP IAM debugging is easier (you only ever need to find the *allow*), at the cost of needing a separate construct for the rare "no, really, never" rules.

**Three layers stacked:** Org Policy says *what can exist*. Deny policy says *what is forbidden regardless of allows*. IAM bindings say *what's allowed*. A request passes only if Org Policy permits the config, no Deny policy denies the action, and at least one IAM binding (with conditions satisfied) allows it. That's the GCP authorization stack in one paragraph.

## What carries into later chapters

The primitives in this notebook show up everywhere. Service accounts are attached to every Cloud Run service (notebook 04), every Compute Engine VM (notebook 03), every GKE workload (notebook 04). Workload Identity Federation is how GitHub Actions deploys anything (notebook 13). `gcp.resourceLocations` is how you enforce the region choice from notebook 01. Deny policies and Security Command Center join the security toolbelt in notebook 11.

The single thing to internalise: **GCP's authorization model is the hierarchy + allow-only IAM + explicit-deny escape hatch + Org Policy + Hierarchical Firewall**. Five layers, each with a different job. Once they click as separate-but-stacked, the rest of GCP's governance surface becomes obvious.